
The selected notebook-13 model is a fixed 50/50 average of sigmoid-calibrated MLP and Extra Trees scores. Final fitting, export and one fixed test evaluation run on the cluster using `cluster/final_ensemble_test.slurm`. See `cluster/FINAL_ENSEMBLE_README.md`.

This notebook **never trains, downloads or predicts**. Download the saved `test_metrics.json` and model ZIP after the cluster job completes. The model package can be attached to a GitHub Release; teammates can load it without retraining.

Current train/test ID and group separation passed audit. Historical project-wide test use remains unverified, so these results must be described with that limitation. No test result has been supplied or invented in this notebook.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src/d1_final_ensemble.py').is_file())
RESULTS = ROOT / 'data/processed/final_ensemble_test/v1'
path = RESULTS / 'test_metrics.json'
result = json.loads(path.read_text()) if path.exists() else None
if result is None:
    print('No final results yet. Submit the cluster job and download its saved metrics.')
else:
    print(result['model'])
    print('Test rows:', result['rows'], '| frauds:', result['fraudulent'], '| groups:', result['groups'])
    display(pd.DataFrame([result['metrics']]).round(4))
    print('Threshold was selected on development inner OOF predictions, not test.')


Calibrated MLP + Extra Trees (50/50)
Test rows: 3441 | frauds: 173 | groups: 885


,pr_auc,roc_auc,precision_at_budget,recall_at_budget,review_share,review_count,review_fraud_caught,review_genuine_flagged,precision_top5pct,recall_top5pct,threshold,precision,recall,f1,share_flagged,fraud_caught,fraud_missed,genuine_flagged,genuine_passed,false_positive_rate
0,0.7848,0.9503,0.7209,0.7168,0.05,172,124,48,0.7209,0.7168,0.2092,0.6772,0.7399,0.7072,0.0549,128,45,61,3207,0.0187


Threshold was selected on development inner OOF predictions, not test.


In [2]:
if result is not None:
    intervals = result['bootstrap']['intervals']
    display(pd.DataFrame([{'metric': name, 'test_estimate': result['metrics'][name],
                           'group_bootstrap_low': bounds['low'],
                           'group_bootstrap_high': bounds['high']}
                          for name, bounds in intervals.items()]).round(4))
    print('Bootstrap valid/requested:', result['bootstrap']['valid'], '/', result['bootstrap']['requested'])
    print(result['history_note'])
    print('Model archive:', result.get('artifact_archive'))
    print('ZIP SHA256:', result.get('artifact_archive_sha256'))


,metric,test_estimate,group_bootstrap_low,group_bootstrap_high
0,pr_auc,0.7848,0.6236,0.8962
1,roc_auc,0.9503,0.9079,0.9799
2,precision,0.6772,0.5374,0.7810
3,recall,0.7399,0.5687,0.8761
4,f1,0.7072,0.5738,0.8098
5,false_positive_rate,0.0187,0.0121,0.0281


Bootstrap valid/requested: 1000 / 1000
Current posting/group separation passed the audit. Earlier project-wide test exposure remains unverified. This evaluation uses the audited current fixed split. Do not describe this as a proven never-accessed project-wide test.
Model archive: /home/e/e1155697/bt4012-fake-job-fraud/data/processed/final_ensemble_test/v1/calibrated_mlp_extra_trees_v1.zip
ZIP SHA256: 81d5bcef760008a1145d99038aee6e3fa2bd6442a483cffef526510528b39320


## Interpretation

Report these as the fixed selected pipeline's test results, not a new model-selection round. Group bootstrap intervals describe uncertainty of this model's metrics; they are not evidence of superiority over an unevaluated comparator. Do not adjust the model or cutoff after reading test scores. Agent integration/evaluation is a separate step.
